# Phase 4: Interactive Visualizations & Dashboards
## USA 2026 FIFA World Cup Economic Impact Analysis

Creating interactive dashboards and professional visualizations for stakeholder presentations.

## Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Load data
historical_wc = pd.read_csv('../data/raw/historical_world_cup_data.csv')
cities = pd.read_csv('../data/raw/usa_host_cities_demographics.csv')
stadiums = pd.read_csv('../data/raw/usa_2026_stadiums.csv')

# Merge cities and stadiums for comprehensive info
merged_cities = cities.merge(stadiums, on='City', how='inner')

# Add calculated impact data (from Phase 3 base case)
# Get dynamic base revenue (from Phase 2 results)
# total_base_revenue = 7621.0 -> replaced
baseline_metrics = pd.read_csv('../data/processed/usa_2026_baseline_metrics.csv')
total_base_revenue = float(baseline_metrics['Estimated_Revenue_2026_Adjusted'].iloc[0])
total_jobs = float(baseline_metrics['Estimated_Employment_2026'].iloc[0])

# Distribute based on Metropolitan GDP and Capacity
merged_cities['Impact_Weight'] = np.log(merged_cities['Metropolitan_GDP_Billions_USD']) * merged_cities['Capacity']
merged_cities['Weight_Normalized'] = merged_cities['Impact_Weight'] / merged_cities['Impact_Weight'].sum()

# Assign Estimated Impact
merged_cities['Estimated_Revenue_Millions'] = merged_cities['Weight_Normalized'] * total_base_revenue
merged_cities['Estimated_Jobs'] = (merged_cities['Weight_Normalized'] * total_jobs).astype(int)

# Use merged_cities as the 'cities' dataframe for the rest of the notebook
cities = merged_cities

print("📊 Phase 4: Visualizations & Dashboards")
print(f"Loaded historical data: {historical_wc.shape}")
print(f"Loaded & Merged cities/stadiums data: {cities.shape}")


📊 Phase 4: Visualizations & Dashboards
Loaded historical data: (7, 18)
Loaded & Merged cities/stadiums data: (8, 25)


## 1. Revenue & Attendance Timeline (Interactive)

In [2]:
# Prepare historical timeline
historical_sorted = historical_wc.sort_values('Year')

# Create interactive line chart
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('World Cup Revenue Over Time', 'Total Attendance Over Time'),
    specs=[[{'secondary_y': False}], [{'secondary_y': False}]],
    vertical_spacing=0.12
)

# Revenue trend
fig.add_trace(
    go.Scatter(
        x=historical_sorted['Year'],
        y=historical_sorted['Revenue_Millions_USD'],
        mode='lines+markers',
        name='Revenue',
        line=dict(color='#1f77b4', width=3),
        marker=dict(size=8),
        hovertemplate='<b>%{x}</b><br>Revenue: $%{y:,.0f}M<extra></extra>'
    ),
    row=1, col=1
)

# Add 2026 projection
fig.add_trace(
    go.Scatter(
        x=[2026],
        y=[7621],
        mode='markers',
        name='2026 Projection',
        marker=dict(size=12, color='#ff7f0e', symbol='star'),
        hovertemplate='<b>2026 (Projected)</b><br>Revenue: $7,621M<extra></extra>'
    ),
    row=1, col=1
)

# Attendance trend
fig.add_trace(
    go.Scatter(
        x=historical_sorted['Year'],
        y=historical_sorted['Total_Attendance'],
        mode='lines+markers',
        name='Attendance',
        line=dict(color='#2ca02c', width=3),
        marker=dict(size=8),
        hovertemplate='<b>%{x}</b><br>Attendance: %{y:,.0f}<extra></extra>'
    ),
    row=2, col=1
)

# Add 2026 projection
fig.add_trace(
    go.Scatter(
        x=[2026],
        y=[670269],
        mode='markers',
        name='2026 Projection',
        marker=dict(size=12, color='#ff7f0e', symbol='star'),
        hovertemplate='<b>2026 (Projected)</b><br>Attendance: 670,269<extra></extra>'
    ),
    row=2, col=1
)

fig.update_xaxes(title_text='Year', row=2, col=1)
fig.update_yaxes(title_text='Revenue (Million USD)', row=1, col=1)
fig.update_yaxes(title_text='Total Attendance', row=2, col=1)

fig.update_layout(
    title='<b>Historical & Projected World Cup Metrics</b>',
    height=700,
    hovermode='x unified',
    template='plotly_white',
    showlegend=True
)

fig.show()
fig.write_html('../data/processed/01_historical_timeline.html')
print("✅ Timeline dashboard saved!")

✅ Timeline dashboard saved!


## 2. Regional Economic Impact Map

In [3]:
# Prepare regional data
regional_data = cities.copy()
regional_data['Revenue_per_Job'] = (regional_data['Estimated_Revenue_Millions'] / regional_data['Estimated_Jobs']).round(3)

# Create treemap of impact
fig = px.treemap(
    regional_data,
    path=[px.Constant('North America 2026'), 'City'],
    values='Estimated_Revenue_Millions',
    color='Estimated_Jobs',
    color_continuous_scale='Viridis',
    title='<b>Regional Economic Impact Across Host Cities</b>',
    hover_data=['Estimated_Revenue_Millions', 'Estimated_Jobs', 'Revenue_per_Job']
)

fig.update_traces(
    hovertemplate=(
        '<b>%{label}</b><br>'
        'Revenue: $%{customdata[0]:,.0f}M<br>'
        'Jobs: %{customdata[1]:,.0f}<br>'
        'Revenue/Job: $%{customdata[2]:,.0f}<extra></extra>'
    )
)

fig.update_layout(
    height=700,
    template='plotly_white'
)

fig.show()
fig.write_html('../data/processed/02_regional_impact_map.html')
print("✅ Regional impact graphic saved!")


✅ Regional impact graphic saved!


## 3. Scenario Analysis Dashboard

In [4]:
# Scenario data
scenarios_data = {
    'Scenario': ['Pessimistic', 'Conservative', 'Base Case', 'Optimistic', 'Very Optimistic'],
    'Revenue_M': [7240, 7621, 7621, 8002, 8231],
    'Attendance': [586485, 628377, 670269, 712161, 754052],
    'Visitors_M': [32.0, 38.4, 48.0, 57.6, 64.0]
}
scenarios_df = pd.DataFrame(scenarios_data)

# Create subplots for scenarios
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Revenue by Scenario', 'Attendance by Scenario', 'Visitor Count by Scenario'),
    specs=[[{}, {}, {}]]
)

colors_scenario = ['#d62728', '#ff7f0e', '#1f77b4', '#2ca02c', '#9467bd']

# Revenue
fig.add_trace(
    go.Bar(
        x=scenarios_df['Scenario'],
        y=scenarios_df['Revenue_M'],
        marker_color=colors_scenario,
        hovertemplate='<b>%{x}</b><br>Revenue: $%{y:,.0f}M<extra></extra>',
        showlegend=False
    ),
    row=1, col=1
)

# Attendance
fig.add_trace(
    go.Bar(
        x=scenarios_df['Scenario'],
        y=scenarios_df['Attendance'],
        marker_color=colors_scenario,
        hovertemplate='<b>%{x}</b><br>Attendance: %{y:,.0f}<extra></extra>',
        showlegend=False
    ),
    row=1, col=2
)

# Visitors
fig.add_trace(
    go.Bar(
        x=scenarios_df['Scenario'],
        y=scenarios_df['Visitors_M'],
        marker_color=colors_scenario,
        hovertemplate='<b>%{x}</b><br>Visitors: %{y:.1f}M<extra></extra>',
        showlegend=False
    ),
    row=1, col=3
)

fig.update_yaxes(title_text='Revenue (Million USD)', row=1, col=1)
fig.update_yaxes(title_text='Total Attendance', row=1, col=2)
fig.update_yaxes(title_text='Visitors (Millions)', row=1, col=3)

fig.update_layout(
    title='<b>Scenario Analysis: Financial Projections</b>',
    height=500,
    template='plotly_white',
    showlegend=False
)

fig.show()
fig.write_html('../data/processed/03_scenario_analysis.html')
print("✅ Scenario analysis dashboard saved!")

✅ Scenario analysis dashboard saved!


## 4. Top Cities Comparison

In [5]:
# Top 10 cities analysis
top_cities = cities.nlargest(10, 'Estimated_Revenue_Millions')

# Create grouped bar chart
fig = go.Figure()

fig.add_trace(go.Bar(
    x=top_cities['City'],
    y=top_cities['Estimated_Revenue_Millions'],
    name='Revenue (Millions USD)',
    marker_color='#1f77b4',
    hovertemplate='<b>%{x}</b><br>Revenue: $%{y:,.0f}M<extra></extra>'
))

# Add jobs on secondary axis
fig.add_trace(go.Bar(
    x=top_cities['City'],
    y=top_cities['Estimated_Jobs']/1000,  # Convert to thousands
    name='Jobs (Thousands)',
    marker_color='#2ca02c',
    yaxis='y2',
    hovertemplate='<b>%{x}</b><br>Jobs: %{y:,.0f}K<extra></extra>'
))

fig.update_layout(
    title='<b>Top 10 Cities: Revenue & Employment Impact</b>',
    xaxis_title='City',
    yaxis=dict(title='Revenue (Million USD)', side='left'),
    yaxis2=dict(title='Jobs Created (Thousands)', overlaying='y', side='right'),
    hovermode='x unified',
    height=500,
    template='plotly_white',
    barmode='group'
)

fig.show()
fig.write_html('../data/processed/04_top_cities_comparison.html')
print("✅ Top cities comparison saved!")

✅ Top cities comparison saved!


## 5. Economic Multiplier Analysis

In [6]:
# Calculate economic efficiency metrics
efficiency_data = cities.copy()
efficiency_data['Jobs_per_Million_Revenue'] = efficiency_data['Estimated_Jobs'] / efficiency_data['Estimated_Revenue_Millions']

# Scatter plot: Revenue vs Jobs
fig = px.scatter(
    efficiency_data.nlargest(15, 'Estimated_Revenue_Millions'),
    x='Estimated_Revenue_Millions',
    y='Estimated_Jobs',
    size='Urban_Population_Millions',
    color='Jobs_per_Million_Revenue',
    hover_name='City',
    hover_data={
        'Estimated_Revenue_Millions': ':.0f',
        'Estimated_Jobs': ':.0f',
        'Urban_Population_Millions': ':.1f',
        'Jobs_per_Million_Revenue': ':.0f'
    },
    title='<b>Economic Multiplier: Revenue vs Employment Impact</b>',
    labels={
        'Estimated_Revenue_Millions': 'Revenue (Million USD)',
        'Estimated_Jobs': 'Jobs Created',
        'Urban_Population_Millions': 'Population',
        'Jobs_per_Million_Revenue': 'Jobs per $M Revenue'
    },
    color_continuous_scale='Blues',
    size_max=40
)

fig.update_traces(
    marker=dict(
        line=dict(width=1, color='white')
    )
)

fig.update_layout(
    height=600,
    template='plotly_white',
    hovermode='closest'
)

fig.show()
fig.write_html('../data/processed/05_economic_multiplier.html')
print("✅ Economic multiplier analysis saved!")

✅ Economic multiplier analysis saved!


## 6. Summary Statistics Dashboard

In [7]:
# Calculate key metrics
total_revenue = cities['Estimated_Revenue_Millions'].sum()
total_jobs = cities['Estimated_Jobs'].sum()
avg_revenue_city = cities['Estimated_Revenue_Millions'].mean()
avg_jobs_city = cities['Estimated_Jobs'].mean()
top_city_revenue = cities['Estimated_Revenue_Millions'].max()
top_city_name = cities.loc[cities['Estimated_Revenue_Millions'].idxmax(), 'City']

print("="*80)
print("📈 KEY FINANCIAL METRICS - USA 2026 FIFA WORLD CUP")
print("="*80)
print(f"\n💰 REVENUE SUMMARY:")
print(f"   Total Projected Revenue: ${total_revenue:,.0f}M")
print(f"   Average Revenue per City: ${avg_revenue_city:,.0f}M")
print(f"   Highest Revenue City: {top_city_name} (${top_city_revenue:,.0f}M)")
print(f"   Revenue Range: ${cities['Estimated_Revenue_Millions'].min():,.0f}M - ${top_city_revenue:,.0f}M")

print(f"\n👷 EMPLOYMENT SUMMARY:")
print(f"   Total Jobs Created: {total_jobs:,.0f}")
print(f"   Average Jobs per City: {avg_jobs_city:,.0f}")
print(f"   Jobs per Million USD Revenue: {total_jobs/total_revenue:,.0f}")

print(f"\n📊 GEOGRAPHIC DISTRIBUTION:")
print(f"   Total Host Cities: {len(cities)}")
print(f"   Top 3 Cities Account for: {(cities.nlargest(3, 'Estimated_Revenue_Millions')['Estimated_Revenue_Millions'].sum()/total_revenue*100):.1f}% of Revenue")
print(f"   Top 5 Cities Account for: {(cities.nlargest(5, 'Estimated_Revenue_Millions')['Estimated_Revenue_Millions'].sum()/total_revenue*100):.1f}% of Revenue")

print(f"\n🎯 EFFICIENCY METRICS:")
print(f"   Revenue Concentration (Herfindahl Index): {sum((cities['Estimated_Revenue_Millions']/total_revenue)**2):.4f}")
print(f"   Std Dev Revenue: ${cities['Estimated_Revenue_Millions'].std():,.0f}M")
print(f"   Coefficient of Variation: {cities['Estimated_Revenue_Millions'].std()/avg_revenue_city:.2f}")

print("="*80)

📈 KEY FINANCIAL METRICS - USA 2026 FIFA WORLD CUP

💰 REVENUE SUMMARY:
   Total Projected Revenue: $4,728M
   Average Revenue per City: $591M
   Highest Revenue City: Los Angeles ($764M)
   Revenue Range: $239M - $764M

👷 EMPLOYMENT SUMMARY:
   Total Jobs Created: 533,417
   Average Jobs per City: 66,677
   Jobs per Million USD Revenue: 113

📊 GEOGRAPHIC DISTRIBUTION:
   Total Host Cities: 8
   Top 3 Cities Account for: 45.3% of Revenue
   Top 5 Cities Account for: 72.5% of Revenue

🎯 EFFICIENCY METRICS:
   Revenue Concentration (Herfindahl Index): 0.1336
   Std Dev Revenue: $166M
   Coefficient of Variation: 0.28


## 7. Export Summary Report

In [8]:
# Export detailed regional analysis
export_data = cities[['City', 'Estimated_Revenue_Millions', 'Estimated_Jobs', 'Urban_Population_Millions']].copy()
export_data = export_data.sort_values('Estimated_Revenue_Millions', ascending=False)
export_data['Revenue_Rank'] = range(1, len(export_data) + 1)
export_data['Jobs_per_Million_Revenue'] = (export_data['Estimated_Jobs'] / export_data['Estimated_Revenue_Millions']).round(0)
export_data['Revenue_per_Capita'] = (export_data['Estimated_Revenue_Millions'] * 1_000_000 / (export_data['Urban_Population_Millions'] * 1_000_000)).round(2)

# Reorder columns
export_data = export_data[[
    'Revenue_Rank', 'City', 'Estimated_Revenue_Millions', 'Estimated_Jobs',
    'Urban_Population_Millions', 'Jobs_per_Million_Revenue', 'Revenue_per_Capita'
]]

# Export to CSV and Excel
export_data.to_csv('../data/processed/regional_impact_summary.csv', index=False)
export_data.to_excel('../data/processed/regional_impact_summary.xlsx', index=False, sheet_name='Regional Impact')

print("✅ Export Summary Report:")
print(export_data.to_string(index=False))
print("\n✅ Files exported:")
print("   - regional_impact_summary.csv")
print("   - regional_impact_summary.xlsx")

✅ Export Summary Report:
 Revenue_Rank        City  Estimated_Revenue_Millions  Estimated_Jobs  Urban_Population_Millions  Jobs_per_Million_Revenue  Revenue_per_Capita
            1 Los Angeles                  763.646904           86157                      13.20                     113.0               57.85
            2     Houston                  711.459018           80269                       7.10                     113.0              100.21
            3     Atlanta                  667.703753           75332                       6.10                     113.0              109.46
            4 Kansas City                  647.502552           73053                       2.15                     113.0              301.16
            5   Monterrey                  639.529396           72154                       5.20                     113.0              122.99
            6 New Orleans                  577.631779           65170                       1.30                     

## 8. Phase 4 Complete

In [9]:
print("\n" + "="*80)
print("✅ PHASE 4: VISUALIZATIONS & DASHBOARDS COMPLETE")
print("="*80)
print("\n📊 Dashboards Created:")
print("   1. Historical Timeline (01_historical_timeline.html)")
print("   2. Regional Impact Map (02_regional_impact_map.html)")
print("   3. Scenario Analysis (03_scenario_analysis.html)")
print("   4. Top Cities Comparison (04_top_cities_comparison.html)")
print("   5. Economic Multiplier (05_economic_multiplier.html)")
print("\n📄 Reports Generated:")
print("   - regional_impact_summary.csv")
print("   - regional_impact_summary.xlsx")
print("\n🎯 Next Phase: Final Report & Executive Summary")
print("="*80)


✅ PHASE 4: VISUALIZATIONS & DASHBOARDS COMPLETE

📊 Dashboards Created:
   1. Historical Timeline (01_historical_timeline.html)
   2. Regional Impact Map (02_regional_impact_map.html)
   3. Scenario Analysis (03_scenario_analysis.html)
   4. Top Cities Comparison (04_top_cities_comparison.html)
   5. Economic Multiplier (05_economic_multiplier.html)

📄 Reports Generated:
   - regional_impact_summary.csv
   - regional_impact_summary.xlsx

🎯 Next Phase: Final Report & Executive Summary
